In this notebook feature engineering and data cleaning are performed. The workflow is:
1. removing some features 
3. handling NaN values
2. adding feature to distinguish the single-hit event from multi-hit event



# 0 - General Ops

In [1]:
import pandas as pd
import numpy as np

import os
from pathlib import Path


In [2]:
# To properly visualization
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# To reset the options to their default values uncomment the following lines and run the cell
# pd.reset_option("display.max_rows")
# pd.reset_option("display.max_columns")
# pd.reset_option("display.max_colwidth")
# pd.reset_option("display.width")

In [3]:
project_dir = Path.cwd().parent
os.chdir(project_dir)
file_path = project_dir / "data" / "Background" / "Engineered_Datasets" 
file_name = "combinedbg.csv"

In [4]:
ds = pd.read_csv(file_path / file_name, header=0)
ds.shape

(324057, 54)

In [5]:
ds.describe(include="all")

,event_index,pixel_ijk,primary_e0_kev,n_voxels,E_sum,E_mean,E_std,E_min,E_max,E_median,E_p10,E_p25,E_p75,E_p90,E_p95,E_p99,E_iqr,E_mad,E_skew,E_kurt,entropy,gini,n_active,frac_active,frac_top1,frac_top3,eff_E_ge_T,com_x,com_y,com_z,rg,lam1,lam2,lam3,linearity,planarity,sphericity,asym_x,asym_y,asym_z,span_x,span_y,span_z,hits_sum,hits_mean_active,corr_E_hits,D_f,D_f_r2,f_occ,L_max_hops,L_max_metric,N_b,L_skel,particle_type
count,324057.000000,324057,3.240570e+05,324057.000000,3.240570e+05,324057.000000,93776.000000,324057.000000,3.240570e+05,324057.000000,324057.000000,324057.000000,324057.000000,324057.000000,324057.000000,3.240570e+05,324057.000000,324057.000000,56221.000000,56221.000000,324057.000000,324057.000000,324057.000000,324057.0,324057.000000,324057.000000,324057.0,324057.000000,324057.000000,324057.000000,324057.000000,324057.000000,324057.000000,324057.000000,93776.000000,93776.000000,93776.000000,324057.000000,324057.000000,324057.000000,324057.000000,324057.000000,324057.000000,324057.000000,324057.0,0.0,6.769200e+04,50639.000000,0.0,324057.000000,324057.000000,324057.000000,324057.000000,324057
unique,NaN,70699,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11
top,NaN,"[9,9,9]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CosmicPhotons
freq,NaN,1663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,152860
mean,98717.561065,NaN,1.637617e+06,2.534097,1.290302e+04,1931.713701,4342.799262,1011.901514,5.100482e+03,1644.385816,1112.692659,1278.406449,2261.893530,2935.819481,3382.900565,4.354344e+03,983.487081,449.499560,0.672945,-0.169721,0.289357,0.028037,2.534097,1.0,0.879039,0.971998,1.0,4.487066,4.454452,4.532519,0.418671,0.806092,0.055047,0.013413,0.917772,0.066917,0.015311,-0.000617,0.000079,0.002401,0.731831,0.735945,0.736016,2.534097,1.0,NaN,5.379954e-01,0.936961,NaN,0.498101,0.577395,0.633950,2.498980,NaN
std,83873.932097,NaN,2.452924e+07,6.643603,9.913632e+04,5078.065590,8995.172299,3992.616448,3.022400e+04,4768.014931,4063.290707,4249.621713,5755.150339,7463.746797,9481.394941,1.816720e+04,3123.506268,1625.740084,1.075869,6.987360,0.559817,0.054361,6.643603,0.0,0.224345,0.095387,0.0,3.175475,3.115990,3.156305,0.836223,2.275469,0.247619,0.078153,0.145645,0.124996,0.041199,0.249144,0.250341,0.251408,1.765916,1.761765,1.764799,6.643603,0.0,NaN,3.974072e-01,0.086823,NaN,1.310938,1.435619,4.876723,8.915356,NaN
min,1.000000,NaN,1.612849e+01,1.000000,1.500616e+01,15.006160,0.003062,15.001220,1.500616e+01,15.006160,15.006160,15.006160,15.006160,15.006160,15.006160,1.500616e+01,0.000000,0.000000,-1.967835,-2.437490,-0.000000,0.000000,1.000000,1.0,0.055853,0.151471,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.012473,0.000000,0.000000,-0.999714,-0.999824,-0.999603,0.000000,0.000000,0.000000,1.000000,1.0,NaN,-1.057477e-16,0.600000,NaN,0.000000,0.000000,0.000000,0.000000,NaN
25%,20434.000000,NaN,1.303700e+02,1.000000,1.247254e+02,120.455395,177.595099,107.848800,1.227150e+02,120.119490,113.125520,117.302175,121.812600,122.424820,122.572340,1.226909e+02,0.000000,0.000000,0.089085,-2.333333,0.000000,0.000000,1.000000,1.0,0.857745,1.000000,1.0,1.433408,1.433463,1.674491,0.000000,0.000000,0.000000,0.000000,0.902038,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.0,NaN,4.121465e-16,0.919369,NaN,0.000000,0.000000,0.000000,0.000000,NaN
50%,79888.000000,NaN,2.153510e+02,1.000000,1.966565e+02,182.054600,2449.724112,148.252260,1.895453e+02,180.108360,158.963200,170.831260,185.851280,188.193250,188.822200,1.894

# 1 - Removing Features

In this section some features are removed. The reasons to remove a particular feature are:
- That feature is not available in real-case scenario: it is only available in a simulation context
- That feature is *redundant* with some other features

## Remove `event_index` feature

In [6]:
if "event_index" in ds.columns.to_list():
    ds = ds.drop(columns=["event_index"])
    print(f"Column 'event_index' removed from dataset, shape is now {ds.shape}.")
else:
    print(f"Column 'event_index' not found in dataset, shape is {ds.shape}.")

Column 'event_index' removed from dataset, shape is now (324057, 53).


## Remove `primary_e0_kev` feature

The `primary_e0_kev` feature is **not available** in real-case scanario: it is the energy of the event at the start of the simulation (before hitting the detector).

In [7]:
if "primary_e0_kev" in ds.columns.to_list():
    ds = ds.drop(columns=["primary_e0_kev"])
    print(f"Column 'primary_e0_kev' removed, shape is now {ds.shape}.")
else:
    print(f"Column 'primary_e0_kev' not found, shape is {ds.shape}.")

Column 'primary_e0_kev' removed, shape is now (324057, 52).


## Remove *total hit*-like feature

The features `n_active`, `hits_sum`, `n_voxels` are redundant: all these features represents the number of activated sub-cubes (detector). Follow a demonstration:

In [8]:
if all(col in ds.columns.to_list() for col in ["n_voxels", "hits_sum","n_active"]):
    ds["hits_sum"] = ds["hits_sum"].astype(int)
    #print(f"Does the columns ['n_voxels', 'hits_sum','n_active'] have the same values? {ds["n_active"].equals(ds["n_voxels"]) and ds["n_active"].equals(ds["hits_sum"])}")
    print(f"n_voxels = hits_sum : {ds['n_voxels'].equals(ds['hits_sum'])}")
    print(f"n_voxels = n_active : {ds['n_voxels'].equals(ds['n_active'])}")
    print(f"hits_sum = n_active : {ds['hits_sum'].equals(ds['n_active'])}")

n_voxels = hits_sum : True
n_voxels = n_active : True
hits_sum = n_active : True


Totally drop `n_active` and `n_voxels`:

In [9]:
if all(col in ds.columns.to_list() for col in ["n_voxels", "hits_sum","n_active"]): 
    ds = ds.drop(columns=["n_voxels", "n_active"])
    print(f"Columns 'n_voxels' and 'n_active' removed, shape is now {ds.shape}.")
else: 
    print(f"Columns 'n_voxels' and 'n_active' not found, shape is {ds.shape}.")  

Columns 'n_voxels' and 'n_active' removed, shape is now (324057, 50).


## Remove single-valued feature

The `hits_mean_active`, `eff_E_ge_T` and `frac_active` feature contain just one unique value across avery sample:

In [10]:
if "hits_mean_active" in ds.columns.to_list():
    print(ds["hits_mean_active"].describe(include="all"))
    print(f"\nNumber of same value ({ds['hits_mean_active'].nunique()}) in 'hits_mean_active': {ds['hits_mean_active'].value_counts().max()} / {ds.shape[0]}")

count    324057.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
Name: hits_mean_active, dtype: float64

Number of same value (1) in 'hits_mean_active': 324057 / 324057


In [11]:
if "frac_active" in ds.columns.to_list():
    print(ds["frac_active"].describe(include="all"))
    print(f"\nNumber of same value ({ds['frac_active'].nunique()}) in 'frac_active': {ds['frac_active'].value_counts().max()} / {ds.shape[0]}")

count    324057.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
Name: frac_active, dtype: float64

Number of same value (1) in 'frac_active': 324057 / 324057


In [12]:
if "eff_E_ge_T" in ds.columns.to_list():
    print(ds["eff_E_ge_T"].describe(include="all"))
    print(f"\nNumber of same value ({ds['eff_E_ge_T'].nunique()}) in 'eff_E_ge_T': {ds['eff_E_ge_T'].value_counts().max()} / {ds.shape[0]}")

count    324057.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
Name: eff_E_ge_T, dtype: float64

Number of same value (1) in 'eff_E_ge_T': 324057 / 324057


Totally drop the columns:

In [13]:
if "hits_mean_active" in ds.columns.to_list():
    ds = ds.drop(columns=["hits_mean_active"])
    print(f"Column 'hits_mean_active' removed, shape is now {ds.shape}.")
if "frac_active" in ds.columns.to_list():
    ds = ds.drop(columns=["frac_active"])
    print(f"Column 'frac_active' removed, shape is now {ds.shape}.")
if "eff_E_ge_T" in ds.columns.to_list():
    ds = ds.drop(columns=["eff_E_ge_T"])
    print(f"Column 'eff_E_ge_T' removed, shape is now {ds.shape}.")

Column 'hits_mean_active' removed, shape is now (324057, 49).
Column 'frac_active' removed, shape is now (324057, 48).
Column 'eff_E_ge_T' removed, shape is now (324057, 47).


# 2 - Handling NaN features

There are feature that contains only NaN values: 

In [14]:
nan_counts = ds.isna().sum()
nan_counts = nan_counts[nan_counts > 0]

nan_perc_df = pd.DataFrame(
    {
        "Column": nan_counts.index, 
        "Total NaN": nan_counts.values, 
        "Percentage": (nan_counts.values / ds.shape[0] * 100).round(2)
    }
).sort_values(by="Percentage", ascending=False).reset_index(drop=True)

nan_perc_df

,Column,Total NaN,Percentage
0,corr_E_hits,324057,100.00
1,f_occ,324057,100.00
2,D_f_r2,273418,84.37
3,E_skew,267836,82.65
4,E_kurt,267836,82.65
5,D_f,256365,79.11
6,E_std,230281,71.06
7,linearity,230281,71.06
8,planarity,230281,71.06
9,sphericity,230281,71.06


Drop those columns with 100% of NaN values:

In [15]:
columns_100perc_nan = nan_perc_df.iloc[:2]["Column"].tolist()  
ds = ds.drop(columns=columns_100perc_nan)
print(f"Columns {columns_100perc_nan} removed, shape is now {ds.shape}.")


Columns ['corr_E_hits', 'f_occ'] removed, shape is now (324057, 45).


## Hits/Shape and NaN correlation

In [16]:
nan_counts = ds.isna().sum()
nan_counts = nan_counts[nan_counts > 0]

onehit_nan_counts = ds[nan_counts.index].apply(lambda x: (x.isna() & (ds['hits_sum'] == 1)).sum())
twohit_nan_counts = ds[nan_counts.index].apply(lambda x: (x.isna() & (ds['hits_sum'] == 2)).sum())
treehit_nan_counts = ds[nan_counts.index].apply(lambda x: (x.isna() & (ds['hits_sum'] == 3)).sum())
morethantreehit_nan_counts = ds[nan_counts.index].apply(lambda x: (x.isna() & (ds['hits_sum'] > 3)).sum())

pd.DataFrame(
    {
        "NaN Features": nan_counts.index,
        "Percentage of NaN values": (nan_counts.values / ds.shape[0] * 100).round(2),
        "NaN with 1 hit": onehit_nan_counts.values,
        "NaN with 2 hits": twohit_nan_counts.values,
        "NaN with 3 hits": treehit_nan_counts.values,
        "NaN with >3 hits": morethantreehit_nan_counts.values 
    }
).sort_values(by=["NaN with 1 hit", "NaN with 2 hits", "NaN with 3 hits", "NaN with >3 hits"], ascending=False).reset_index(drop=True)

,NaN Features,Percentage of NaN values,NaN with 1 hit,NaN with 2 hits,NaN with 3 hits,NaN with >3 hits
0,D_f_r2,84.37,230281,37555,4820,762
1,E_skew,82.65,230281,37555,0,0
2,E_kurt,82.65,230281,37555,0,0
3,D_f,79.11,230281,22713,2923,448
4,E_std,71.06,230281,0,0,0
5,linearity,71.06,230281,0,0,0
6,planarity,71.06,230281,0,0,0
7,sphericity,71.06,230281,0,0,0


Most of NaN are due to the low quantity of hits per event. Let's visualize the correlation between the NaN values and the number of hits (=1, =2 or > 2)

In [17]:
one_hit = ds[ds["hits_sum"] == 1]
less2_hit = ds[ds["hits_sum"] <= 2]
multi_hit = ds[ds["hits_sum"] > 2]

print(f"Number of events with 1 hit: {one_hit.shape[0]} ({one_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with 2 or less hit: {less2_hit.shape[0]} ({less2_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with multiple hits: {multi_hit.shape[0]} ({multi_hit.shape[0]/ds.shape[0]*100:.2f}%)")

bg = ds["particle_type"].unique()
less2_hit_counts = [less2_hit[less2_hit["particle_type"] == b].shape[0] for b in bg]
less2_hit_percentages = [less2_hit[less2_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in bg]
pd.DataFrame(
    {
        "Background": bg,
        "One Hit count": [f"{one_hit[one_hit['particle_type'] == b].shape[0]} / {ds[ds['particle_type'] == b].shape[0]}" for b in bg],
        "One Hit percentage": np.round([one_hit[one_hit['particle_type'] == b].shape[0]/ds[ds['particle_type'] == b].shape[0]*100 for b in bg], 2),
        "Two or less Hits count": [f"{less2_hit_counts[i]} / {ds[ds['particle_type'] == bg[i]].shape[0]}" for i in range(len(bg))],
        #"One Hit count": f"{one_hit_counts} / {ds[ds["particle_type"] == bg].shape[0]}",
        "Two or less Hits percentage": np.round(less2_hit_percentages, 2),
    }
)

Number of events with 1 hit: 230281 (71.06%)
Number of events with 2 or less hit: 267836 (82.65%)
Number of events with multiple hits: 56221 (17.35%)


,Background,One Hit count,One Hit percentage,Two or less Hits count,Two or less Hits percentage
0,SecondaryProtonsUpward,5902 / 11901,49.59,7249 / 11901,60.91
1,PrimaryPositrons,1 / 4,25.00,3 / 4,75.00
2,CosmicPhotons,137476 / 152860,89.94,150935 / 152860,98.74
3,PrimaryProtons,3575 / 18461,19.37,5710 / 18461,30.93
4,AlbedoPhotons,65667 / 86862,75.60,78072 / 86862,89.88
5,PrimaryElectrons,25 / 148,16.89,33 / 148,22.30
6,SecondaryElectrons,3134 / 8730,35.90,4523 / 8730,51.81
7,AtmosphericNeutrons,1364 / 2161,63.12,1711 / 2161,79.18
8,PrimaryAlphas,624 / 2781,22.44,911 / 2781,32.76
9,SecondaryProtonsDownward,5486 / 11060,49.60,6746 / 11060,60.99


⚠️ Datasets must be balanced!

In [18]:
nan_counts = ds.isna().sum()
nan_counts = nan_counts[nan_counts > 0]

columns_with_nan = nan_counts.index

less2_hit_nan_counts = less2_hit[columns_with_nan].isna().sum()
multi_hit_nan_counts = multi_hit[columns_with_nan].isna().sum()
less2_hit_nan_counts_percent = np.round(less2_hit[columns_with_nan].isna().sum() / nan_counts[columns_with_nan] * 100, 2)
multi_hit_nan_counts_percent = np.round(multi_hit[columns_with_nan].isna().sum() / nan_counts[columns_with_nan] * 100, 2)

nan_report = pd.DataFrame({
    "NaN Count": nan_counts,
    "Two or less Hits NaN Count": less2_hit_nan_counts,
    "Multi Hits NaN Count": multi_hit_nan_counts,
    "Two or less Hits NaN percentage": less2_hit_nan_counts_percent,
    "Multi Hits NaN percentage": multi_hit_nan_counts_percent
})

nan_report.index.name = "Column"
nan_report.reset_index()

,Column,NaN Count,Two or less Hits NaN Count,Multi Hits NaN Count,Two or less Hits NaN percentage,Multi Hits NaN percentage
0,E_std,230281,230281,0,100.00,0.00
1,E_skew,267836,267836,0,100.00,0.00
2,E_kurt,267836,267836,0,100.00,0.00
3,linearity,230281,230281,0,100.00,0.00
4,planarity,230281,230281,0,100.00,0.00
5,sphericity,230281,230281,0,100.00,0.00
6,D_f,256365,252994,3371,98.69,1.31
7,D_f_r2,273418,267836,5582,97.96,2.04


The geometrical features and some energy features are obviously correlated with the number of hits: to define some of these features, at least 2 hits are required (e.g., standard deviation). In particular:
- E_std needs at least 2 hits,
- linearity, planarity and sphericity need at least 2,3+ hits
- E_skew (skewness) at lest 3 hits
- E_kurt 

Two different approach are possible: 
- setting those feature to a default value (e.g., 0) if the number of hits is equal to 1 (or <=2)
- create differents sub-datasets:
    - One-hit dataset which does not contain the NaN features
    - *"Less than 2"*-hit dataset
    - Multi-hit dataset and handle separately the remaining NaNs 

# 3 - Distinguish events by number of hit

In [19]:
one_hit = ds[ds["hits_sum"] == 1]
less2_hit = ds[ds["hits_sum"] <= 2]
multi_hit = ds[ds["hits_sum"] > 2]

print(f"Number of events with 1 hit: {one_hit.shape[0]} ({one_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with 2 or less hit: {less2_hit.shape[0]} ({less2_hit.shape[0]/ds.shape[0]*100:.2f}%)")
print(f"Number of events with multiple hits: {multi_hit.shape[0]} ({multi_hit.shape[0]/ds.shape[0]*100:.2f}%)")

bg = ds["particle_type"].unique()
one_hit_counts = [one_hit[one_hit["particle_type"] == b].shape[0] for b in bg]
one_hit_percentages = [one_hit[one_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in bg]
less2_hit_counts = [less2_hit[less2_hit["particle_type"] == b].shape[0] for b in bg]
less2_hit_percentages = [less2_hit[less2_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in bg]
multi_hit_counts = [multi_hit[multi_hit["particle_type"] == b].shape[0] for b in bg]
multi_hit_percentages = [multi_hit[multi_hit["particle_type"] == b].shape[0]/ds[ds["particle_type"] == b].shape[0]*100 for b in bg]
pd.DataFrame(
    {
        "Background": bg,
        "One Hit count": [f"{one_hit_counts[i]} / {ds[ds['particle_type'] == bg[i]].shape[0]}" for i in range(len(bg))],
        "One Hit percentage": np.round(one_hit_percentages, 2),
        "Two or less Hits count": [f"{less2_hit_counts[i]} / {ds[ds['particle_type'] == bg[i]].shape[0]}" for i in range(len(bg))],
        "Two or less Hits percentage": np.round(less2_hit_percentages, 2),
        "Multiple Hits count": [f"{multi_hit_counts[i]} / {ds[ds['particle_type'] == bg[i]].shape[0]}" for i in range(len(bg))],
        "Multiple Hits percentage": np.round(multi_hit_percentages, 2)
    }
).sort_values(by="Two or less Hits percentage", ascending=False)

Number of events with 1 hit: 230281 (71.06%)
Number of events with 2 or less hit: 267836 (82.65%)
Number of events with multiple hits: 56221 (17.35%)


,Background,One Hit count,One Hit percentage,Two or less Hits count,Two or less Hits percentage,Multiple Hits count,Multiple Hits percentage
2,CosmicPhotons,137476 / 152860,89.94,150935 / 152860,98.74,1925 / 152860,1.26
4,AlbedoPhotons,65667 / 86862,75.60,78072 / 86862,89.88,8790 / 86862,10.12
7,AtmosphericNeutrons,1364 / 2161,63.12,1711 / 2161,79.18,450 / 2161,20.82
1,PrimaryPositrons,1 / 4,25.00,3 / 4,75.00,1 / 4,25.00
9,SecondaryProtonsDownward,5486 / 11060,49.60,6746 / 11060,60.99,4314 / 11060,39.01
0,SecondaryProtonsUpward,5902 / 11901,49.59,7249 / 11901,60.91,4652 / 11901,39.09
6,SecondaryElectrons,3134 / 8730,35.90,4523 / 8730,51.81,4207 / 8730,48.19
10,SecondaryPositrons,7027 / 29089,24.16,11943 / 29089,41.06,17146 / 29089,58.94
8,PrimaryAlphas,624 / 2781,22.44,911 / 2781,32.76,1870 / 2781,67.24
3,PrimaryProtons,3575 / 18461,19.37,5710 / 18461,30.93,12751 / 18461,69.07


Two possible approach:
- mantain one single dataset with a *flag*-like feature which distinguish between single or multi hits 
- divide two the dataset in two sub-datasets

NaN values must be handled!

In [20]:
one_hit = ds[ds["hits_sum"] == 1]
less2_hit = ds[ds["hits_sum"] <= 2]
multi_hit = ds[ds["hits_sum"] > 2]

### One-hit Dataset (and *"2 or less"*-hit)

In [21]:
print(f"\"Raw\" One hit dataset shape: {one_hit.shape}")
print(f"\n\nNaN values in one hit dataset:")
oh_nan_val = one_hit.isna().sum()
oh_nan_val = oh_nan_val[oh_nan_val > 0]
print(oh_nan_val)

"Raw" One hit dataset shape: (230281, 45)


NaN values in one hit dataset:
E_std         230281
E_skew        230281
E_kurt        230281
linearity     230281
planarity     230281
sphericity    230281
D_f           230281
D_f_r2        230281
dtype: int64


In [22]:
print(f"\"Raw\" Less 2 hit dataset shape: {less2_hit.shape}")
print(f"\n\nNaN values in less 2 hit dataset:")
l2_nan_val = less2_hit.isna().sum()
l2_nan_val = l2_nan_val[l2_nan_val > 0]
print(l2_nan_val)

"Raw" Less 2 hit dataset shape: (267836, 45)


NaN values in less 2 hit dataset:
E_std         230281
E_skew        267836
E_kurt        267836
linearity     230281
planarity     230281
sphericity    230281
D_f           252994
D_f_r2        267836
dtype: int64


Just simply drop this features

In [23]:
## less one hit dataset cleaning
columns_to_drop = oh_nan_val.index.tolist()
if columns_to_drop:
    print(f"\nDropping columns with NaN values in one hit dataset: {columns_to_drop}")
    one_hit = one_hit.drop(columns=columns_to_drop)

    print(f"NaN values in one hit dataset after dropping NaN values: {one_hit.isna().sum().sum()}")

print(f'"Cleaned" One hit dataset shape: {one_hit.shape}')

## less 2 hit dataset cleaning
columns_to_drop = l2_nan_val.index.tolist()
if columns_to_drop:
    print(f"\nDropping columns with NaN values in less 2 hit dataset: {columns_to_drop}")
    less2_hit = less2_hit.drop(columns=columns_to_drop)

    print(f"NaN values in less 2 hit dataset after dropping NaN values: {less2_hit.isna().sum().sum()}")

print(f'"Cleaned" Less 2 hit dataset shape: {less2_hit.shape}')


Dropping columns with NaN values in one hit dataset: ['E_std', 'E_skew', 'E_kurt', 'linearity', 'planarity', 'sphericity', 'D_f', 'D_f_r2']
NaN values in one hit dataset after dropping NaN values: 0
"Cleaned" One hit dataset shape: (230281, 37)

Dropping columns with NaN values in less 2 hit dataset: ['E_std', 'E_skew', 'E_kurt', 'linearity', 'planarity', 'sphericity', 'D_f', 'D_f_r2']
NaN values in less 2 hit dataset after dropping NaN values: 0
"Cleaned" Less 2 hit dataset shape: (267836, 37)


In [24]:
one_hit.describe(include="all")

,pixel_ijk,E_sum,E_mean,E_min,E_max,E_median,E_p10,E_p25,E_p75,E_p90,E_p95,E_p99,E_iqr,E_mad,entropy,gini,frac_top1,frac_top3,com_x,com_y,com_z,rg,lam1,lam2,lam3,asym_x,asym_y,asym_z,span_x,span_y,span_z,hits_sum,L_max_hops,L_max_metric,N_b,L_skel,particle_type
count,230281,230281.000000,230281.000000,230281.000000,230281.000000,230281.000000,230281.000000,230281.000000,230281.000000,230281.000000,230281.000000,230281.000000,230281.0,230281.0,230281.0,2.302810e+05,230281.0,230281.0,230281.000000,230281.000000,230281.000000,230281.0,230281.0,230281.0,230281.0,230281.0,230281.0,230281.0,230281.0,230281.0,230281.0,230281.0,230281.0,230281.0,230281.0,230281.0,230281
unique,500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11
top,"[9,9,9]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CosmicPhotons
freq,1663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,137476
mean,NaN,1035.893953,1035.893953,1035.893953,1035.893953,1035.893953,1035.893953,1035.893953,1035.893953,1035.893953,1035.893953,1035.893953,0.0,0.0,0.0,3.295653e-17,1.0,1.0,4.487739,4.474994,4.537730,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
std,NaN,4356.031325,4356.031325,4356.031325,4356.031325,4356.031325,4356.031325,4356.031325,4356.031325,4356.031325,4356.031325,4356.031325,0.0,0.0,0.0,7.894120e-17,0.0,0.0,3.312642,3.250235,3.291091,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
min,NaN,15.006160,15.006160,15.006160,15.006160,15.006160,15.006160,15.006160,15.006160,15.006160,15.006160,15.006160,0.0,0.0,-0.0,0.000000e+00,1.0,1.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
25%,NaN,115.481310,115.481310,115.481310,115.481310,115.481310,115.481310,115.481310,115.481310,115.481310,115.481310,115.481310,0.0,0.0,0.0,0.000000e+00,1.0,1.0,1.000000,1.000000,1.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
50%,NaN,150.244060,150.244060,150.244060,150.244060,150.244060,150.244060,150.244060,150.244060,150.244060,150.244060,150.244060,0.0,0.0,0.0,0.000000e+00,1.0,1.0,5.000000,5.000000,5.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN
75%,NaN,247.377550,247.377550,247.377550,247.377550,247.377550,247.377550,247.377550,247.377550,247.377550,247.377550,247.377550,0.0,0.0,0.0,0.000000e+00,1.0,1.0,8.000000,8.000000,8.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN


In [25]:
less2_hit.describe(include="all")

,pixel_ijk,E_sum,E_mean,E_min,E_max,E_median,E_p10,E_p25,E_p75,E_p90,E_p95,E_p99,E_iqr,E_mad,entropy,gini,frac_top1,frac_top3,com_x,com_y,com_z,rg,lam1,lam2,lam3,asym_x,asym_y,asym_z,span_x,span_y,span_z,hits_sum,L_max_hops,L_max_metric,N_b,L_skel,particle_type
count,267836,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,2.678360e+05,267836.000000,267836.0,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,2.678360e+05,2.678360e+05,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,267836.000000,267836.0,267836.000000,267836
unique,15477,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11
top,"[9,9,9]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CosmicPhotons
freq,1663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,150935
mean,NaN,1538.001550,1214.323091,1054.310918,1374.335263,1214.323091,1086.313352,1134.317004,1294.329177,1342.332829,1358.334046,1371.135020,160.012173,160.012173,0.072898,1.669237e-02,0.963277,1.0,4.487278,4.470029,4.525750,0.115802,0.148484,3.915510e-18,9.010580e-20,-0.000734,0.000130,0.002622,0.130031,0.131491,0.135512,1.140216,0.084802,0.101337,0.0,0.290617,NaN
std,NaN,5969.792879,4577.569370,4276.107529,5135.396444,4577.569370,4312.653010,4390.441865,4829.035131,5006.868963,5070.184531,5122.205800,1172.414085,1172.414085,0.191853,4.873314e-02,0.104570,0.0,3.276752,3.216141,3.255764,0.367525,0.836984,5.254861e-17,3.339548e-18,0.159329,0.161710,0.162747,0.530245,0.528922,0.545332,0.347212,0.278587,0.341057,0.0,0.909070,NaN
min,NaN,15.006160,15.006160,15.001220,15.006160,15.006160,15.006160,15.006160,15.006160,15.006160,15.006160,15.006160,0.000000,0.000000,-0.000000,0.000000e+00,0.500007,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,-0.998816,-0.999239,-0.999486,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.0,0.000000,NaN
25%,NaN,118.267735,114.885120,111.911765,116.500020,114.885120,112.782935,113.536460,115.864622,116.257030,116.382733,116.481748,0.000000,0.000000,-0.000000,0.000000e+00,1.000000,1.0,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.0,0.000000,NaN
50%,NaN,161.543755,154.465360,149.113005,158.295275,154.465360,151.024015,152.220565,156.562460,157.670010,158.019775,158.230585,0.000000,0.000000,-0.000000,0.000000e+00,1.000000,1.0,4.485767,4.795149,5.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.0,0.000000,NaN
75%,NaN,320.110565,283.942865,259.679542,298.527907,283.942865,271.639436,277.371740,291.014286,295.399000,297.077618,298.206298,0.000000,0.000000,0.000000,2.220446e-16,1.000000,1.0,8.000000,8.000000,8.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.0,0.000000,NaN


Almost every faeture is meaningless if the number of hits is equal to 1 (or <=2). Drop those feature:

In [26]:
one_hit = one_hit[["pixel_ijk", "E_sum", "com_x", "com_y", "com_z", "hits_sum", "particle_type"]]
less2_hit = less2_hit[["pixel_ijk", "E_sum", "com_x", "com_y", "com_z", "hits_sum", "particle_type"]]

### Multi-hit Dataset

In [27]:
print(f"\"Raw\" Multi hits dataset shape: {multi_hit.shape}")
print(f"\n\nNaN values in multi hits dataset:")
mh_nan_val = multi_hit.isna().sum()
mh_nan_val = mh_nan_val[mh_nan_val > 0]
print(mh_nan_val)

"Raw" Multi hits dataset shape: (56221, 45)


NaN values in multi hits dataset:
D_f       3371
D_f_r2    5582
dtype: int64


<!-- To compute the skewness and the kurtosis coeff. at least 3 hits are needed. To avoid to create a third dataset, and since there are almost ~40% missing values, we can decide to totally drop these features. -->

R2 index may be dropped. For the Fractal Dimension we decide to impute a default value 0, preserving the meaning of the feature: a too small to be computed fractal dimension may be set to 0.  


In [28]:
# mh_cols_drop = ["E_skew", "E_kurt", "D_f_r2"]
mh_cols_drop = ["D_f_r2"]

if all(col in multi_hit.columns.to_list() for col in mh_cols_drop):
    print(f"\nDropping columns with NaN values in multi hits dataset: {mh_cols_drop}")
    multi_hit = multi_hit.drop(columns=mh_cols_drop)

    print(f"NaN values in multi hits dataset after dropping NaN values: {multi_hit.isna().sum().sum()}")

# impute 0 to D_f
if "D_f" in multi_hit.columns.to_list():
    multi_hit["D_f"] = multi_hit["D_f"].fillna(0)
    print(f"NaN values in 'D_f' column after imputation: {multi_hit['D_f'].isna().sum()}")

print(f'"Cleaned" Multi hits dataset shape: {multi_hit.shape}')


Dropping columns with NaN values in multi hits dataset: ['D_f_r2']
NaN values in multi hits dataset after dropping NaN values: 3371
NaN values in 'D_f' column after imputation: 0
"Cleaned" Multi hits dataset shape: (56221, 44)


### Summary and saving 

In [29]:
print(f"Final shape of one hit dataset: {one_hit.shape}")
print(f"Final shape of less 2 hit dataset: {less2_hit.shape}")
print(f"Final shape of multi hits dataset: {multi_hit.shape}")

Final shape of one hit dataset: (230281, 7)
Final shape of less 2 hit dataset: (267836, 7)
Final shape of multi hits dataset: (56221, 44)


In [30]:
if "multi_hit" in one_hit.columns.to_list():
    one_hit = one_hit.drop(columns=["multi_hit"])
    print(f"Column 'multi_hit' removed from one_hit dataset, shape is now {one_hit.shape}.")
if "multi_hit" in less2_hit.columns.to_list():
    less2_hit = less2_hit.drop(columns=["multi_hit"])
    print(f"Column 'multi_hit' removed from less2_hit dataset, shape is now {less2_hit.shape}.")
if "multi_hit" in multi_hit.columns.to_list():
    multi_hit = multi_hit.drop(columns=["multi_hit"])
    print(f"Column 'multi_hit' removed from multi_hit dataset, shape is now {multi_hit.shape}.")

In [31]:
one_hit.to_csv(file_path / "one_hit_dataset.csv", index=False)
less2_hit.to_csv(file_path / "less2_hit_dataset.csv", index=False)
multi_hit.to_csv(file_path / "multi3plus_hit_dataset.csv", index=False)